# Validation tests

In [132]:
#installation packages
!pip install -q cupy-cuda12x
%pip install QuantLib

In [133]:
#libraries
import QuantLib as ql
import cupy as cp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.stats import norm
import time


## SciPy Implementation Regular

This code implements a CPU-based semi-analytical Heston European call-option pricer

Input: Heston model params
Output: Theoretical price of European call option

In [134]:
def heston_charac_function_cpu(
        u,
        S0,
        T,
        r,
        v0,
        kappa,
        theta,
        sigma,
        rho
):
    i = 1j
    d = np.sqrt(
        (rho*sigma*i*u-kappa)**2
        + sigma**2 * (i * u + u**2)
    )
    g = (kappa - rho * sigma * i * u - d)/(kappa - rho * sigma * i * u + d)
    C = (r * i * u * T
        + (kappa * theta / sigma**2)
        *((kappa - rho * sigma * i * u - d) * T
            - 2
            * np.log(
                (1-g * np.exp(-d*T))
                / (1-g)
            )
        )
    )
    D = (
        (kappa - rho * sigma * i * u - d)
        / sigma ** 2
        * (
            (1 - np.exp(-d * T))
            / (1 - g * np.exp(-d * T))
        )
    )
    return np.exp(
        C
        + D * v0
        + i * u * np.log(S0)
    )

def heston_probability_cpu(
        j,
        S0,
        K,
        T,
        r,
        v0,
        kappa,
        theta,
        sigma,
        rho,
        upper_limit=100 #runs convergence tests with 50, 100, 200, etc.
):
    phi_minus_i = heston_charac_function_cpu(
        -1j,
        S0,
        T,
        r,
        v0,
        kappa,
        theta,
        sigma,
        rho
    )
    def integrand(u):
        phi_u = heston_charac_function_cpu(
            u,
            S0,
            T,
            r,
            v0,
            kappa,
            theta,
            sigma,
            rho
        )
        if j == 1:
            phi_shifted = heston_charac_function_cpu(
                u-1j,
                S0,
                T,
                r,
                v0,
                kappa,
                theta,
                sigma,
                rho
            )
            value = (
                np.exp(-1j * u * np.log(K))
                * phi_shifted
                / (1j * u * phi_minus_i)
            )
        else:
            value = (
                np.exp(-1j * u * np.log(K))
                * phi_u
                /(1j * u)
            )
        return np.real(value)
    integral, error = quad(
        integrand,
        1e-8,
        100,
        limit=500
    )
    return 0.5 + integral/np.pi

def heston_price_scipy_cpu(
        S0,
        K,
        T,
        r,
        v0,
        kappa,
        theta,
        sigma,
        rho
):
    P1 = heston_probability_cpu(
        1,
        S0,
        K,
        T,
        r,
        v0,
        kappa,
        theta,
        sigma,
        rho
    )

    P2 = heston_probability_cpu(
        2,
        S0,
        K,
        T,
        r,
        v0,
        kappa,
        theta,
        sigma,
        rho
    )

    price = (
        S0 * P1 - K * np.exp(-r * T) * P2
    )

    return price

In [135]:
#testing CPU implementation
S0 = 100
K = 100
T = 1.0
r = 0.03
v0 = 0.04
kappa = 2.0
theta = 0.04
sigma = 0.30
rho = -0.70

price = heston_price_scipy_cpu(
    S0,
    K,
    T,
    r,
    v0,
    kappa,
    theta,
    sigma,
    rho
)

#testing characteristic function
phi0 = heston_charac_function_cpu(
    0.0,
    S0,
    T,
    r,
    v0,
    kappa,
    theta,
    sigma,
    rho
)

#testing p1 and p2 values
p1 = heston_probability_cpu(
    1,
    S0,
    K,
    T,
    r,
    v0,
    kappa,
    theta,
    sigma,
    rho
)
p2 = heston_probability_cpu(
    2,
    S0,
    K,
    T,
    r,
    v0,
    kappa,
    theta,
    sigma,
    rho
)

#testing no arbitrage bounds
lower_bound = max(
    S0 - K * np.exp(-r * T),
    0
)
upper_bound = S0
assert lower_bound <= price <= upper_bound

print("Test Heston Price: ", price)
print("Test Characteristic Function: ", phi0) #should always be 1+0j
print("Test P1: ", p1) #0\leq p1 \leq 1
print("Test P2: ", p2) #0 \leq p2 \leq 1
print("No aribtrage test passed")

Test Heston Price:  9.242521061494699
Test Characteristic Function:  (1+0j)
Test P1:  0.6556511539247819
Test P2:  0.5803787269238658
No aribtrage test passed


## SciPy Implementation GPU

In [136]:
'''
def heston_charac_function_gpu(
        u,
        S0,
        T,
        r,
        v0,
        kappa,
        theta,
        sigma,
        rho
):
    i = 1j
    d = cp.sqrt(
        (rho*sigma*i*u-kappa)**2
        + sigma**2 * (i * u + u**2)
    )
    g = (kappa - rho * sigma * i * u - d)/(kappa - rho * sigma * i * u + d)
    C = (r * i * u * T
        + (kappa * theta / sigma **2)
        *(
            (kappa - rho * sigma * i * u - d) * T
            - 2 * cp.log((1-g * cp.exp(-d*T))/ (1-g))))
    D = (
        (kappa - rho * sigma * i * u - d)
        / sigma ** 2
        * (
            (1 - cp.exp(-d * T))
            / (1 - g * cp.exp(-d * T))
        )
    )
    return cp.exp(
        C
        + D * v0
        + i * u * cp.log(S0)
    )

def heston_probability_gpu(
        j,
        S0,
        K,
        T,
        r,
        v0,
        kappa,
        theta,
        sigma,
        rho,
        N=20_000,
        u_max=100
):
    u = cp.linspace(
        1e-8,
        u_max,
        N,
        dtype=cp.float64
    )

    phi_minus_i = heston_charac_function_gpu(
        cp.asarray(-1j),
        S0,
        T,
        r,
        v0,
        kappa,
        theta,
        sigma,
        rho
    )
    if j == 1:
      phi_shifted = heston_charac_function_gpu(
          u-1j,
          S0,
          T,
          r,
          v0,
          kappa,
          theta,
          sigma,
          rho
      )
      integrand = cp.real(
          cp.exp(-1j * u * cp.log(K)) * phi_shifted / (1j * u * phi_minus_i)
      )
    else:
      phi_u = heston_charac_function_gpu(
          u,
          S0,
          T,
          r,
          v0,
          kappa,
          theta,
          sigma,
          rho
      )
      integrand = cp.real(
          cp.exp(-1j * u * cp.log(K)) * phi_u / (1j * u)
      )
    integral = cp.trapezoid(integrand, u)
    probability = (0.5 + integral/cp.pi)
    return probability

def heston_price_gpu(
        S0,
        K,
        T,
        r,
        v0,
        kappa,
        theta,
        sigma,
        rho,
        N=20_000,
        u_max=100
):
    S0 = cp.asarray(S0, dtype=cp.float64),
    K = cp.asarray(K, dtype=cp.float64),
    T = cp.asarray(T, dtype=cp.float64),
    r = cp.asarray(r, dtype=cp.float64),
    v0 = cp.asarray(v0, dtype=cp.float64),
    kappa = cp.asarray(kappa, dtype=cp.float64),
    theta = cp.asarray(theta, dtype=cp.float64),
    sigma = cp.asarray(sigma, dtype=cp.float64),
    rho = cp.asarray(rho, dtype=cp.float64),

    P1 = heston_probability_gpu(
        1,
        S0,
        K,
        T,
        r,
        v0,
        kappa,
        theta,
        sigma,
        rho,
        N=N,
        u_max=u_max
    )

    P2 = heston_probability_gpu(
        2,
        S0,
        K,
        T,
        r,
        v0,
        kappa,
        theta,
        sigma,
        rho,
        N=N,
        u_max=u_max
    )

    price = (
        S0 * P1 - K * cp.exp(-r * T) * P2
    )

    return float(cp.asnumpy(price))
'''

'\ndef heston_charac_function_gpu(\n        u,\n        S0,\n        T,\n        r,\n        v0,\n        kappa,\n        theta,\n        sigma,\n        rho\n):\n    i = 1j\n    d = cp.sqrt(\n        (rho*sigma*i*u-kappa)**2\n        + sigma**2 * (i * u + u**2)\n    )\n    g = (kappa - rho * sigma * i * u - d)/(kappa - rho * sigma * i * u + d)\n    C = (r * i * u * T\n        + (kappa * theta / sigma **2)\n        *(\n            (kappa - rho * sigma * i * u - d) * T\n            - 2 * cp.log((1-g * cp.exp(-d*T))/ (1-g))))\n    D = (\n        (kappa - rho * sigma * i * u - d)\n        / sigma ** 2\n        * (\n            (1 - cp.exp(-d * T))\n            / (1 - g * cp.exp(-d * T))\n        )\n    )\n    return cp.exp(\n        C\n        + D * v0\n        + i * u * cp.log(S0)\n    )\n\ndef heston_probability_gpu(\n        j,\n        S0,\n        K,\n        T,\n        r,\n        v0,\n        kappa,\n        theta,\n        sigma,\n        rho,\n        N=20_000,\n        u_max=10

## QuantLib Implementation

In [137]:
#reusing quantlib function from before
def heston_quantlib_prices(
    S0, #stock price today
    K, #strike price
    T, #time to maturity (in years)
    r, #risk free interest rate
    v0, #variance today
    kappa, #speed of mean reversion
    theta, #long run variance
    sigma, #volatility of variance
    rho, #correlation between stock and variance
    q=0.0 #dividend yield
):
    #date setup
    today = ql.Date.todaysDate()
    ql.Settings.instance().evaluationDate = today

    #conversion of T into maturity date
    maturity_date = today + int(round(T * 365))

    #day count conversion
    day_count = ql.Actual365Fixed()

    #spot price
    spot = ql.QuoteHandle(
        ql.SimpleQuote(S0)
    )

    #risk free rate
    risk_free_curve = ql.YieldTermStructureHandle(
        ql.FlatForward(today,r,day_count)
    )

    dividend_curve = ql.YieldTermStructureHandle(
        ql.FlatForward(today,q,day_count)
    )

    heston_process = ql.HestonProcess(
        risk_free_curve,
        dividend_curve,
        spot,
        v0,
        kappa,
        theta,
        sigma,
        rho
    )

    #heston model
    heston_model = ql.HestonModel(heston_process)

    #analytical heston pricing engine
    engine = ql.AnalyticHestonEngine(heston_model)

    #European call
    payoff = ql.PlainVanillaPayoff(
        ql.Option.Call,
        K
    )

    #
    exercise = ql.EuropeanExercise(
        maturity_date
    )

    option = ql.VanillaOption(
        payoff,
        exercise
    )

    option.setPricingEngine(engine)

    return option.NPV()

## Black Scholes
Implementation for validation testing later.

In [ ]:
def black_scholes_call(
    S0,
    K,
    T,
    r,
    sigma_b_s,
    q=0.0
):
  d1 = (
      np.log(S0/K)
      + (
          r - q + 0.5 * sigma_b_s ** 2
      ) * T
  ) / (sigma_b_s * np.sqrt(T)
  )

  d2 = (
      d1 - sigma_b_s * np.sqrt(T)
  )
  price = (
      S0 * np.exp(-q * T) * norm.cdf(d1)
      - K * np.exp(-r * T) * norm.cdf(d2)
  )
  return price

# Comparison Function
Compares both SciPy and QuantLib implementations

## Validation Test 1:
Reproducing result from known example

In [ ]:
params = {
    "S0": 100.0,
    "K": 100.0,
    "T": 1.0,
    "r": 0.03,
    "v0": 0.04,
    "kappa": 2.0,
    "theta": 0.04,
    "sigma": 0.30,
    "rho": -0.70
}

ql_price = heston_quantlib_prices(**params)
scipy_price = heston_price_scipy(**params)
print(f"QuantLib price: {ql_price:.8f}")
print(f"Scipy Price: {scipy_price: .10f}")

In [ ]:
absolute_error = abs(
    ql_price - scipy_price
)

relative_error = (
    absolute_error / abs(ql_price)
)

print(f"Absolute error: {absolute_error: .10e}")
print(f"Relative error: {relative_error: .10e}")

In [ ]:
print(f"Relative Error: {relative_error * 100: .8f}%")

In [ ]:
def compare_pricers(params):
    ql_price = heston_quantlib_prices(**params)
    scipy_price = heston_price_scipy(**params)
    abs_error = abs(ql_price - scipy_price)
    rel_error = (abs_error / max(abs(ql_price), 1e-12))
    return{
        **params,
        "quantlib_price": ql_price,
        "scipy_price": scipy_price,
        "absolute_error": abs_error,
        "relative_error": rel_error
    }

result = compare_pricers(params)
result

In [ ]:
strikes = np.arange(60,141, 10)
results = []

for K in strikes:
    test_params = {
        "S0": 100.0,
        "K": float(K),
        "T": 1.0,
        "r": 0.03,
        "v0": 0.04,
        "kappa": 2.0,
        "theta": 0.04,
        "sigma": 0.30,
        "rho": -0.70
    }
    result = compare_pricers(test_params)

    results.append(result)

In [ ]:
df_strikes = pd.DataFrame(results)
df_strikes

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    df_strikes["K"].to_numpy(),
    df_strikes["quantlib_price"].to_numpy(),
    marker="o",
    label="QuantLib"
)

plt.plot(
    df_strikes["K"].to_numpy(),
    df_strikes["scipy_price"].to_numpy(),
    marker="x",
    linestyle="--",
    label="SciPy"
)

plt.xlabel("Strike K")
plt.ylabel("European Call Price")
plt.title("Heston Pricing Validation: QuantLib vs SciPy")

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    df_strikes["K"].to_numpy(),
    df_strikes["absolute_error"].to_numpy(),
    marker="o"
)

plt.xlabel("Strike K")
plt.ylabel("Absolute Error")
plt.title("Absolute Pricing Error: QuantLib vs SciPy")

plt.grid(True)
plt.show()

In [ ]:
#testing different maturities
maturities = [
    0.1,
    0.25,
    0.5,
    1.0,
    2.0,
    3.0,
    5.0
]

results = []

for T in maturities:
    test_params = {
        "S0": 100.0,
        "K": 100.0,
        "T": T,
        "r": 0.03,
        "v0": 0.04,
        "kappa": 2.0,
        "theta": 0.04,
        "sigma": 0.30,
        "rho": -0.70
    }
    results.append(compare_pricers(test_params))

df_maturities = pd.DataFrame(results)

df_maturities[
    [
        "T",
        "quantlib_price",
        "scipy_price",
        "absolute_error",
        "relative_error"
    ]
]

In [ ]:
#random heston params
rng = np.random.default_rng(42)
n_test = 100

results = []

for _ in range(n_test):
    test_params = {
        "S0": 100.0,
        "K": rng.uniform(70,130),
        "T": rng.uniform(0.1, 3.0),
        "r": rng.uniform(0.0,0.08),
        "v0": rng.uniform(0.01, 0.16),
        "kappa": rng.uniform(0.5, 5.0),
        "theta": rng.uniform(0.01, 0.16),
        "sigma": rng.uniform(0.1, 1.0),
        "rho": rng.uniform(-0.9, -0.1)
    }

    try:
        result = compare_pricers(test_params)
        results.append(results)
    except Exception as e:
        print("Failed parameter set:")
        print(test_params)
        print(e)

In [ ]:
df_validation = pd.DataFrame(results)
df_validation.head()

In [ ]:
print("Number of successful tests:", len(df_validation))
print("Mean absolute error:",
      df_validation["absolute_error"].mean())
print("Median absolute error:",
      df_validation["absolute_error"].median())
print("Max absolute error:",
      df_validation["absolute_error"].max())
print("Mean relative error:",
      df_validation["relative_error"].mean())
print("Median relative error:",
      df_validation["relative_error"].median())
print("Max relative error:",
      df_validation["relative_error"].max())

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(
    df_validation["quantlib_price"],
    df_validation["scipy_price"],
    alpha=0.6
)

minimum = min(
    df_validation["quantlib_price"].min(),
    df_validation["scipy_price"].min()
)

maximum = max(
    df_validation["quantlib_price"].max(),
    df_validation["scipy_price"].max()
)

plt.plot(
    [minimum,minimum],
    [minimum,maximum],
    "--",
    label="Perfect Agreement"
)

plt.xlabel("Quantlib Heston Price")
plt.ylabel("SciPy Heston Price")

plt.title(
    "Validation of Heston Pricing Implementation"
)

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
worst_index = (
    df_validation["absolute_error"].idxmax()
)

worst_case = df_validation.loc[worst_index]

worst_case

In [ ]:
#validation 2: basic no arbitrage bound
lower_bound = max(
    params["S0"] - params["K"] * np.exp(-params["r"] * params["T"]),
    0
)

upper_bound = params["S0"]

print(f"Lower bound: {lower_bound:.6f}")
print(f"Heston price: {ql_price: .6f}")
print(f"Upper bound: {upper_bound:.6f}")

assert lower_bound <= ql_price <= upper_bound

print("No-arbitrage bounds passed")

In [ ]:
#validation 3: price should decrease with strike

strikes = np.linspace(60, 140, 81)

prices = []

for K in strikes:
    price = heston_quantlib_prices(
        S0 = 100,
        K = K,
        T = 1.0,
        r = 0.03,
        v0 = 0.04,
        kappa = 2.0,
        theta = 0.04,
        sigma = 0.30,
        rho = -0.70
    )

    prices.append(price)


#plotting validation
plt.figure(figsize=(8, 5))
plt.plot(strikes, prices)
plt.xlabel("Strike K")
plt.ylabel("Call Price")
plt.title("Heston Call Price vs. Strike")

plt.grid(True)
plt.show()


In [ ]:
assert np.all(np.diff(prices) <= 1e-10)

In [ ]:
#validation 4: price behaviour with spot
spots = np.linspace(60, 140, 81)

spot_prices = []

for S0 in spots:
    price = heston_quantlib_prices(
        S0=S0,
        K=100,
        T=1.0,
        r=0.03,
        v0=0.04,
        kappa=2.0,
        theta=0.04,
        sigma=0.30,
        rho=-0.70
    )
    spot_prices.append(price)

#plotting
plt.figure(figsize=(8,5))
plt.plot(spots,spot_prices)
plt.xlabel("Spot Price $S_0$")
plt.ylabel("Call Price")
plt.title("Heston Call Price vs. Spot Price")

plt.grid(True)
plt.show()


#test
assert np.all(np.diff(spot_prices)>=-1e-10)

Validation Test: Comparison of Quantlib and CPU against Black-Scholes

In [ ]:
from scipy.stats import norm

In [ ]:
S0 = 100.0
K = 100.0
T = 1.0
r = 0.03

sigma_b_s = 0.20
v0 = sigma_b_s ** 2
theta = sigma_b_s ** 2
kappa = 2.0
sigma = 1e-4
rho = 0.0

bs_price = black_scholes_call(
    S0,
    K,
    T,
    r,
    sigma_b_s
)

cpu_price = heston_price_scipy(
    S0,
    K,
    T,
    r,
    v0,
    kappa,
    theta,
    sigma,
    rho
)
ql_price = heston_quantlib_prices(
    S0,
    K,
    T,
    r,
    v0,
    kappa,
    theta,
    sigma,
    rho
)